# 🩺 Breast Cancer Multimodal Detection
**Modalities:** Ultrasound · Histopathology · Chest X-ray  
**Architecture:** EfficientNetB0 × 3 → Feature Fusion → Dense(512) → Dense(256) → Sigmoid  
**Datasets:** BUSI · BreakHis · Chest X-ray (Kaggle)

---
### How to run:
1. Run **Step 1** (install)
2. Run **Step 2** (clone repo)
3. Upload `kaggle.json` → run **Step 3**
4. Run **Step 4** (download datasets)
5. Run **Step 5** (organize)
6. Run **Step 6** (train multimodal model) ← main step
7. Run **Step 7** (train tabular models if not done)
8. Run **Step 8 + 9** (start Flask app)

## Step 1 — Install Dependencies

In [ ]:
!pip install -q flask flask-cors pillow scikit-learn numpy==1.26.4 pandas joblib kaggle pyngrok tensorflow

## Step 2 — Clone / Update GitHub Repo

In [ ]:
import os, sys

REPO_URL = 'https://github.com/fadicrazy/AI-PROJECT.git'
REPO_DIR = '/content/AI-PROJECT'
PROJ_DIR = '/content/AI-PROJECT/breast_cancer_ai'

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL}
else:
    !cd {REPO_DIR} && git pull
    print('Repo already exists — pulled latest.')

sys.path.append(PROJ_DIR)
os.chdir(PROJ_DIR)
print('Files:', os.listdir(PROJ_DIR))

## Step 3 — Kaggle API Setup
1. Go to **kaggle.com → Account → API → Create New Token**
2. Upload the downloaded `kaggle.json` file here in Colab (left panel → upload icon)
3. Then run this cell

In [ ]:
import shutil, os
src = '/content/kaggle.json'
dst = os.path.expanduser('~/.kaggle/kaggle.json')
if os.path.exists(src):
    os.makedirs(os.path.dirname(dst), exist_ok=True)
    shutil.copy(src, dst)
    os.chmod(dst, 0o600)
    print('✅ Kaggle API configured!')
else:
    print('❌ kaggle.json not found! Upload it to /content/ first.')

## Step 4 — Download Datasets

In [ ]:
DATA_ROOT = '/content/datasets'
os.makedirs(DATA_ROOT, exist_ok=True)

print('[1/3] BUSI Ultrasound...')
!kaggle datasets download -d aryashah2003/breast-ultrasound-images-dataset -p {DATA_ROOT}/raw_us --unzip -q

print('[2/3] BreakHis Histopathology...')
!kaggle datasets download -d forderation/breakhis -p {DATA_ROOT}/raw_histo --unzip -q

print('[3/3] Chest X-ray...')
!kaggle datasets download -d paultimothymooney/chest-xray-pneumonia -p {DATA_ROOT}/raw_xray --unzip -q

print('\n✅ All downloads done!')

## Step 5 — Organize Datasets (train / val / test split)

In [ ]:
import random, shutil
from pathlib import Path

DATA_ROOT = '/content/datasets'

def organize_split(src_benign, src_malignant, dest_dir,
                   val_ratio=0.15, test_ratio=0.10, seed=42):
    random.seed(seed)
    for label, src in [('benign', src_benign), ('malignant', src_malignant)]:
        p = Path(src)
        if not p.exists():
            print(f'  ⚠️  Not found: {src}')
            continue
        files = [f for f in p.rglob('*') if f.suffix.lower() in ['.png','.jpg','.jpeg']]
        random.shuffle(files)
        n = len(files)
        nv, nt = int(n*val_ratio), int(n*test_ratio)
        splits = {'train':files[:n-nv-nt], 'val':files[n-nv-nt:n-nt], 'test':files[n-nt:]}
        for split, imgs in splits.items():
            out = Path(dest_dir)/split/label
            out.mkdir(parents=True, exist_ok=True)
            for img in imgs: shutil.copy(img, out/img.name)
        print(f'  {label}: {len(splits["train"])} train | {len(splits["val"])} val | {len(splits["test"])} test')

print('Organizing Ultrasound (BUSI)...')
organize_split(f'{DATA_ROOT}/raw_us/Dataset_BUSI_with_GT/benign',
               f'{DATA_ROOT}/raw_us/Dataset_BUSI_with_GT/malignant',
               f'{DATA_ROOT}/ultrasound')

print('Organizing Histopathology (BreakHis)...')
organize_split(f'{DATA_ROOT}/raw_histo/BreaKHis_v1/histology_slides/breast/benign/SOB',
               f'{DATA_ROOT}/raw_histo/BreaKHis_v1/histology_slides/breast/malignant/SOB',
               f'{DATA_ROOT}/histopathology')

print('Organizing Chest X-ray...')
organize_split(f'{DATA_ROOT}/raw_xray/chest_xray/train/NORMAL',
               f'{DATA_ROOT}/raw_xray/chest_xray/train/PNEUMONIA',
               f'{DATA_ROOT}/chest_xray')

print('\n✅ Done!')

## Step 6 — Train Multimodal Model
⚠️ **Enable GPU first!** Runtime → Change runtime type → T4 GPU  
This takes ~30-40 minutes.

In [ ]:
!cd /content/AI-PROJECT/breast_cancer_ai && python multimodal_train.py

## Step 7 — Train Tabular + Original CNN Models (if not done)

In [ ]:
!cd /content/AI-PROJECT/breast_cancer_ai && python train_models.py
!cd /content/AI-PROJECT/breast_cancer_ai && python train_image_model.py

## Step 8 — Start Flask App

In [ ]:
import base64, json, os, random, sys
from io import BytesIO
import joblib, numpy as np, pandas as pd
from flask import Flask, jsonify, render_template, request
from flask_cors import CORS
from PIL import Image
from sklearn.datasets import load_breast_cancer

PROJ_DIR = '/content/AI-PROJECT/breast_cancer_ai'
sys.path.append(PROJ_DIR)
os.chdir(PROJ_DIR)

from agents.agent import CancerAgent

MODELS_DIR    = os.path.join(PROJ_DIR, 'models')
TEMPLATES_DIR = os.path.join(PROJ_DIR, 'templates')

app = Flask(__name__, template_folder=TEMPLATES_DIR)
CORS(app, resources={r'/api/*': {'origins': '*'}})
app.config['MAX_CONTENT_LENGTH'] = 16 * 1024 * 1024

cnn_model        = None
multimodal_model = None
agent            = CancerAgent()

with open(os.path.join(PROJ_DIR, 'model_results.json')) as f:
    MODEL_RESULTS = json.load(f)
with open(os.path.join(PROJ_DIR, 'cnn_metrics.json')) as f:
    CNN_METRICS = json.load(f)

_mm = os.path.join(PROJ_DIR, 'multimodal_metrics.json')
MULTIMODAL_METRICS = json.load(open(_mm)) if os.path.exists(_mm) else {}

SCALER        = joblib.load(os.path.join(MODELS_DIR, 'scaler.pkl'))
FEATURE_NAMES = joblib.load(os.path.join(MODELS_DIR, 'feature_names.pkl'))
MODELS = {
    'Random Forest':       joblib.load(os.path.join(MODELS_DIR, 'random_forest.pkl')),
    'Gradient Boosting':   joblib.load(os.path.join(MODELS_DIR, 'gradient_boosting.pkl')),
    'SVM':                 joblib.load(os.path.join(MODELS_DIR, 'svm.pkl')),
    'Logistic Regression': joblib.load(os.path.join(MODELS_DIR, 'logistic_regression.pkl')),
    'KNN':                 joblib.load(os.path.join(MODELS_DIR, 'knn.pkl')),
}
DATA          = load_breast_cancer(as_frame=True)
DF            = DATA.frame
FEATURE_MEANS = DF[FEATURE_NAMES].mean()
FEATURE_STDS  = DF[FEATURE_NAMES].std()
FEATURE_EXAMPLES = {n: float(round(FEATURE_MEANS[n], 3)) for n in FEATURE_NAMES}
TARGET_LABELS = {0: 'Malignant', 1: 'Benign'}
print('✅ All imports done!')

In [ ]:
def lazy_load_cnn():
    global cnn_model
    if cnn_model is None:
        import tensorflow as tf
        cnn_model = tf.keras.models.load_model(os.path.join(MODELS_DIR, 'cnn_model.keras'))
    return cnn_model

def lazy_load_multimodal():
    global multimodal_model
    if multimodal_model is None:
        import tensorflow as tf
        p = os.path.join(MODELS_DIR, 'multimodal_model.keras')
        if not os.path.exists(p):
            raise FileNotFoundError('Multimodal model not found. Run Step 6 first.')
        multimodal_model = tf.keras.models.load_model(p)
    return multimodal_model

def normalize_features(fv):
    return SCALER.transform(np.array(fv, dtype=float).reshape(1, -1))

def risk_level(p):
    return 'High' if p >= 0.85 else ('Moderate' if p >= 0.65 else 'Low')

def flagged_features(fmap):
    flags = []
    for n, v in fmap.items():
        if n not in FEATURE_MEANS: continue
        d = abs(v - FEATURE_MEANS[n]) / (FEATURE_STDS[n] + 1e-9)
        if d >= 1.3:
            flags.append({'feature': n, 'value': float(v),
                          'comparison': ('higher than' if v > FEATURE_MEANS[n] else 'lower than') + ' average',
                          'deviation': float(round(d, 2))})
    return flags[:6]

def build_prediction_payload(model_name, features):
    scaled = normalize_features(features)
    mdl    = MODELS[model_name]
    proba  = mdl.predict_proba(scaled)[0]
    pred   = int(mdl.predict(scaled)[0])
    label  = TARGET_LABELS[pred]
    prob   = float(round(proba[1] if pred == 1 else 1 - proba[1], 4))
    risk   = risk_level(1 - proba[1] if pred == 0 else proba[1])
    fmap   = {n: float(features[i]) for i, n in enumerate(FEATURE_NAMES)}
    return {'model': model_name, 'label': label, 'prediction': pred,
            'probabilities': {'Benign': float(round(proba[1], 4)), 'Malignant': float(round(proba[0], 4))},
            'confidence': float(round(max(proba) * 100, 2)), 'risk_level': risk,
            'flagged_features': flagged_features(fmap),
            'analysis': agent.generate_report(label, prob, risk, fmap)}

def preprocess_image(image_bytes, size=64, channels=1):
    mode = 'L' if channels == 1 else 'RGB'
    img  = Image.open(BytesIO(image_bytes)).convert(mode).resize((size, size), Image.LANCZOS)
    arr  = np.asarray(img, dtype=np.float32) / 255.0
    arr  = arr.reshape(1, size, size, channels)
    th   = img.resize((128, 128), Image.LANCZOS)
    buf  = BytesIO(); th.save(buf, format='PNG')
    prev = base64.b64encode(buf.getvalue()).decode('utf-8')
    return arr, prev

print('✅ Helper functions ready!')

In [ ]:
@app.route('/')
def home(): return render_template('index.html')

@app.route('/api/health')
def health(): return jsonify({'status': 'ok', 'message': 'Breast Cancer AI API running.'})

@app.route('/api/feature_names')
def feature_names():
    return jsonify({'feature_names': list(FEATURE_NAMES), 'feature_examples': FEATURE_EXAMPLES})

@app.route('/api/dataset_stats')
def get_dataset_stats():
    counts = DF['target'].value_counts().to_dict()
    return jsonify({'samples': int(DF.shape[0]), 'features': len(FEATURE_NAMES),
                    'classes': list(DATA.target_names),
                    'distribution': {TARGET_LABELS[k]: int(v) for k, v in counts.items()},
                    'feature_groups': {'mean': [n for n in FEATURE_NAMES if 'mean' in n],
                                       'se':   [n for n in FEATURE_NAMES if 'se' in n],
                                       'worst':[n for n in FEATURE_NAMES if 'worst' in n]}})

@app.route('/api/model_results')
def get_model_results(): return jsonify(MODEL_RESULTS)

@app.route('/api/compare_models')
def compare_models():
    return jsonify({'models': sorted(MODEL_RESULTS['models'], key=lambda x: x['accuracy'], reverse=True)})

@app.route('/api/cnn_metrics')
def cnn_metrics_route(): return jsonify(CNN_METRICS)

@app.route('/api/multimodal_metrics')
def multimodal_metrics_route(): return jsonify(MULTIMODAL_METRICS)

@app.route('/api/dataset_samples')
def dataset_samples():
    count = min(request.args.get('count', 5, type=int), 20)
    idxs  = np.random.choice(DF.shape[0], size=count, replace=False)
    return jsonify({'samples': [{'index': int(i), 'features': DF.iloc[i][FEATURE_NAMES].tolist(),
                                  'target': TARGET_LABELS[int(DF.iloc[i]['target'])]} for i in idxs]})

@app.route('/api/predict', methods=['POST'])
def predict():
    p  = request.get_json(force=True)
    mn = p.get('model') or 'Random Forest'
    ft = p.get('features')
    if not ft or len(ft) != len(FEATURE_NAMES):
        return jsonify({'error': 'features must be a list of 30 values'}), 400
    try: return jsonify(build_prediction_payload(mn, ft))
    except Exception as e: return jsonify({'error': str(e)}), 400

@app.route('/api/predict_image', methods=['POST'])
def predict_image():
    if 'image' not in request.files:
        return jsonify({'error': 'No image file provided.'}), 400
    try:
        content = request.files['image'].read()
        model   = lazy_load_cnn()
        arr, prev = preprocess_image(content, 64, 1)
        proba = float(model.predict(arr, verbose=0)[0][0])
        label = 'Benign' if proba >= 0.5 else 'Malignant'
        risk  = risk_level(proba)
        return jsonify({'label': label, 'probability': round(proba, 4),
                        'confidence': round(max(proba, 1-proba)*100, 2), 'risk_level': risk,
                        'probabilities': {'Benign': round(proba,4), 'Malignant': round(1-proba,4)},
                        'analysis': agent.generate_report(label, round(proba,4), risk, {}),
                        'thumbnail': f'data:image/png;base64,{prev}'})
    except Exception as e: return jsonify({'error': str(e)}), 500

@app.route('/api/predict_multimodal', methods=['POST'])
def predict_multimodal():
    ZERO = np.zeros((1, 224, 224, 3), dtype=np.float32)
    inputs, previews = [], {}
    for modality in ['ultrasound', 'histopathology', 'chest_xray']:
        f = request.files.get(modality)
        if f and f.filename:
            arr, prev = preprocess_image(f.read(), 224, 3)
            inputs.append(arr); previews[modality] = f'data:image/png;base64,{prev}'
        else:
            inputs.append(ZERO); previews[modality] = None
    if all(np.array_equal(x, ZERO) for x in inputs):
        return jsonify({'error': 'Provide at least one image.'}), 400
    try:
        model = lazy_load_multimodal()
        proba = float(model.predict(inputs, verbose=0)[0][0])
        label = 'Benign' if proba >= 0.5 else 'Malignant'
        risk  = risk_level(proba)
        return jsonify({'label': label, 'probability': round(proba, 4),
                        'confidence': round(max(proba, 1-proba)*100, 2), 'risk_level': risk,
                        'probabilities': {'Benign': round(proba,4), 'Malignant': round(1-proba,4)},
                        'modalities_provided': [m for m in ['ultrasound','histopathology','chest_xray'] if previews[m]],
                        'thumbnails': previews,
                        'analysis': agent.generate_report(label, round(proba,4), risk, {}),
                        'model': 'Multimodal EfficientNetB0 (US + Histo + CXR)'})
    except FileNotFoundError as e: return jsonify({'error': str(e)}), 503
    except Exception as e: return jsonify({'error': str(e)}), 500

@app.route('/api/predict_sample')
def predict_sample():
    idx = random.randint(0, DF.shape[0]-1)
    s   = DF.iloc[idx]; ft = s[FEATURE_NAMES].tolist()
    res = {'sample_index': int(idx), 'features': {n: float(s[n]) for n in FEATURE_NAMES},
           'target': TARGET_LABELS[int(s['target'])],
           'predictions': [build_prediction_payload(n, ft) for n in MODELS]}
    res['agent_report'] = agent.generate_report_for_sample(res)
    return jsonify(res)

@app.route('/api/agent/chat', methods=['POST'])
def agent_chat():
    p = request.get_json(force=True)
    q = p.get('question', '').strip()
    if not q: return jsonify({'error': 'Question required.'}), 400
    return jsonify({'question': q, 'answer': agent.chat(q, p.get('context', {}))})

@app.errorhandler(413)
def too_large(e): return jsonify({'error': 'File too large. Max 16MB.'}), 413

print('✅ All routes registered!')

## Step 9 — Launch with ngrok
⚠️ Replace `YOUR_NGROK_TOKEN` with your token from https://dashboard.ngrok.com

In [ ]:
from pyngrok import ngrok

ngrok.set_auth_token('YOUR_NGROK_TOKEN')  # ← Replace this!
public_url = ngrok.connect(5000)
print('🌐 App URL:', public_url)
print('📡 Multimodal API:', str(public_url).split('"')[1] + '/api/predict_multimodal')

In [ ]:
app.run(host='0.0.0.0', port=5000, debug=False, use_reloader=False)